In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/spotify-tracks-dataset.csv', index_col=0)
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

# Remove duplicates
df = df.drop_duplicates()
print(f"After removing duplicates: {df.shape}")

# Drop rows with missing values
df = df.dropna()
print(f"After dropping nulls: {df.shape}")

# Convert explicit to int
df['explicit'] = df['explicit'].astype(int)

# Duration in minutes instead of ms
df['duration_min'] = df['duration_ms'] / 60000

print("\n✅ Basic cleaning done")
df.head()

After removing duplicates: (113550, 20)
After dropping nulls: (113549, 20)

✅ Basic cleaning done


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,duration_min
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,0,0.676,0.4610,1,...,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic,3.844433
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,0,0.420,0.1660,1,...,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic,2.493500
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,0,0.438,0.3590,0,...,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic,3.513767
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,0,0.266,0.0596,0,...,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic,3.365550
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,0,0.618,0.4430,2,...,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic,3.314217


In [2]:
# Mood score: combines valence and energy
df['mood_score'] = (df['valence'] * 0.6 + df['energy'] * 0.4).round(3)

# Acoustic score: how acoustic vs electronic
df['acoustic_score'] = (df['acousticness'] * 0.7 + (1 - df['energy']) * 0.3).round(3)

# Danceability-energy interaction
df['dance_energy'] = (df['danceability'] * df['energy']).round(3)

# Popularity tier
df['popularity_tier'] = pd.cut(df['popularity'],
    bins=[-1, 20, 40, 60, 80, 100],
    labels=['unknown', 'low', 'medium', 'high', 'viral']
)

# Genre avg popularity (how popular is this genre on average)
genre_avg = df.groupby('track_genre')['popularity'].mean()
df['genre_avg_popularity'] = df['track_genre'].map(genre_avg).round(2)

# Artist avg popularity
artist_avg = df.groupby('artists')['popularity'].mean()
df['artist_avg_popularity'] = df['artists'].map(artist_avg).round(2)

print("✅ Features engineered:")
print(f"  mood_score, acoustic_score, dance_energy")
print(f"  popularity_tier, genre_avg_popularity, artist_avg_popularity")
print(f"\nFinal shape: {df.shape}")
df[['track_name','artists','track_genre','popularity','popularity_tier',
    'mood_score','dance_energy','genre_avg_popularity']].head(10)

✅ Features engineered:
  mood_score, acoustic_score, dance_energy
  popularity_tier, genre_avg_popularity, artist_avg_popularity

Final shape: (113549, 27)


,track_name,artists,track_genre,popularity,popularity_tier,mood_score,dance_energy,genre_avg_popularity
0,Comedy,Gen Hoshino,acoustic,73,high,0.613,0.312,42.48
1,Ghost - Acoustic,Ben Woodward,acoustic,55,medium,0.227,0.070,42.48
2,To Begin Again,Ingrid Michaelson;ZAYN,acoustic,57,medium,0.216,0.157,42.48
3,Can't Help Falling In Love,Kina Grannis,acoustic,71,high,0.110,0.016,42.48
4,Hold On,Chord Overstreet,acoustic,82,viral,0.277,0.274,42.48
5,Days I Will Remember,Tyrone Wells,acoustic,58,medium,0.592,0.331,42.48
6,Say Something,A Great Big World;Christina Aguilera,acoustic,74,high,0.105,0.060,42.48
7,I'm Yours,Jason Mraz,acoustic,80,high,0.605,0.312,42.48
8,Lucky,Jason Mraz;Colbie Caillat,acoustic,74,high,0.567,0.259,42.48
9,Hunger,Ross Copperman,acoustic,56,medium,0.370,0.279,42.48


In [3]:
genre_stats = df.groupby('track_genre').agg(
    avg_popularity   = ('popularity', 'mean'),
    track_count      = ('track_name', 'count'),
    avg_danceability = ('danceability', 'mean'),
    avg_energy       = ('energy', 'mean'),
    avg_valence      = ('valence', 'mean'),
    avg_mood         = ('mood_score', 'mean'),
).reset_index().round(3)

genre_stats = genre_stats.sort_values('avg_popularity', ascending=False)

print("Top 20 genres by popularity:")
print(genre_stats.head(20).to_string(index=False))

# Save
genre_stats.to_csv('../data/genre_stats.csv', index=False)
print("\n✅ genre_stats.csv saved")

Top 20 genres by popularity:
      track_genre  avg_popularity  track_count  avg_danceability  avg_energy  avg_valence  avg_mood
         pop-film          59.280          999             0.597       0.604        0.529     0.559
            k-pop          56.964          998             0.648       0.676        0.557     0.604
            chill          53.705          999             0.664       0.426        0.404     0.413
              sad          52.379         1000             0.692       0.462        0.422     0.438
           grunge          49.583          999             0.457       0.803        0.400     0.562
           indian          49.529          999             0.592       0.567        0.463     0.505
            anime          48.767          999             0.538       0.674        0.435     0.530
              emo          48.128         1000             0.599       0.670        0.441     0.533
              pop          47.903          993             0.630       

In [4]:
artist_stats = df.groupby('artists').agg(
    avg_popularity   = ('popularity', 'mean'),
    track_count      = ('track_name', 'count'),
    avg_danceability = ('danceability', 'mean'),
    avg_energy       = ('energy', 'mean'),
    avg_mood         = ('mood_score', 'mean'),
    genres           = ('track_genre', lambda x: ', '.join(x.unique()[:3]))
).reset_index().round(3)

artist_stats = artist_stats[artist_stats['track_count'] >= 2]
artist_stats = artist_stats.sort_values('avg_popularity', ascending=False)

print("Top 20 artists by popularity:")
print(artist_stats.head(20)[['artists','avg_popularity','track_count','genres']].to_string(index=False))

artist_stats.to_csv('../data/artist_stats.csv', index=False)
print("\n✅ artist_stats.csv saved")

Top 20 artists by popularity:
                        artists  avg_popularity  track_count                    genres
           Sam Smith;Kim Petras         100.000            2                dance, pop
                  Manuel Turizo          98.000            4     latin, latino, reggae
     Bad Bunny;Chencho Corleone          97.000            4     latin, latino, reggae
        Bad Bunny;Bomba Estéreo          94.500            4     latin, latino, reggae
                   Harry Styles          92.000            3                       pop
    Rauw Alejandro;Lyanno;Brray          91.000            2             latin, latino
                      Luar La L          90.500            4     latin, latino, reggae
            Bad Bunny;Tony Dize          90.000            3 latino, reggae, reggaeton
       Bad Bunny;Rauw Alejandro          90.000            3 latino, reggae, reggaeton
                    Nicki Minaj          89.000            3            dance, hip-hop
   Lost Frequ

In [5]:
# Save the full cleaned and featured dataset
df.to_csv('../data/spotify_clean.csv', index=False)
print(f"✅ spotify_clean.csv saved — {df.shape[0]:,} tracks, {df.shape[1]} columns")
print(f"\nFinal columns: {list(df.columns)}")

✅ spotify_clean.csv saved — 113,549 tracks, 27 columns

Final columns: ['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre', 'duration_min', 'mood_score', 'acoustic_score', 'dance_energy', 'popularity_tier', 'genre_avg_popularity', 'artist_avg_popularity']
